In [ ]:
from pathlib import Path

ROOT = Path.cwd()
assert (ROOT / "src").exists(), f"Run this notebook from the assignment folder: {ROOT}"
assert (ROOT / "data").exists(), "Missing data/ folder. Did you clone the repo correctly?"

from src.geo_math import haversine_km, destination_point

print("Dallas → Houston (km):",
      haversine_km(32.7767, -96.7970, 29.7604, -95.3698))

print("Destination:",
      destination_point(32.7767, -96.7970, 180, 100))

In [ ]:
"""
**************************************************************************
*
* Author: Bryce Koch
* Email: bryce.koch@my.msutexas.edu
* Label: Project_01_Code
* Title: Missile Geometry 101 - Milestone 1
* Course: CMPS 5993
* Semester: Spring 2026
*
* Description:
* This milestone establishes the spatial foundation for all subsequent
* milestones. A world borders shapefile is loaded and converted to
* GeoJSON features, which are rendered as a layer on an interactive
* Folium map. The WDO base location is plotted as a labeled marker.
* The projection file is inspected to confirm the coordinate reference
* system before any spatial operations are performed. A layer control
* is added to allow toggling map layers on and off.
*
* Usage:
* Run as a Jupyter notebook cell before any other milestone.
* All subsequent milestones depend on BASE_LAT, BASE_LON, and m
* defined here.
*
* Files:
* notebook.ipynb             : this notebook cell
* src/io_shapefile.py        : shapefile_to_features, read_prj_if_exists
* src/viz_map.py             : make_base_map, add_geojson_layer, add_base_marker
* src/geo_math.py            : haversine_km, destination_point, trajectory_points
* data/world_borders/        : world borders shapefile and supporting files
*
**************************************************************************
"""

from pathlib import Path
from src.io_shapefile import shapefile_to_features, read_prj_if_exists
from src.viz_map import make_base_map, add_geojson_layer, add_base_marker
from src.geo_math import haversine_km, destination_point, trajectory_points
import folium

# Path to the world borders shapefile directory and target file
DATA_DIR = Path("data/world_borders")
SHP_PATH = DATA_DIR / "world_borders.shp"
print(SHP_PATH)

# Convert shapefile to a list of GeoJSON-compatible feature dicts
# id_field=None means no specific attribute is used as a feature identifier
features = shapefile_to_features(SHP_PATH, id_field=None)

# Read the projection file to confirm the CRS before any spatial operations
# A missing PRJ file could cause silent coordinate misalignment later
prj = read_prj_if_exists(SHP_PATH)
print("PRJ exists:", prj is not None)
if prj:
    print(prj[:200], "...\n")

# WDO base coordinates — Dallas, TX
# These are used as the fixed command center for all spatial analysis
BASE_LAT, BASE_LON = 32.7767, -96.7970

# Initialize the Folium map centered on the base at a world-level zoom
m = make_base_map(BASE_LAT, BASE_LON, zoom=2, tiles="OpenStreetMap")

# Add world borders as a toggleable GeoJSON layer
# tooltip_field=None disables hover labels on country polygons
add_geojson_layer(m, features, name="World Borders", tooltip_field=None)

# Place a labeled marker at the WDO base location
add_base_marker(m, BASE_LAT, BASE_LON, label="WDO Base")

# Add layer control so map layers can be toggled on and off interactively
folium.LayerControl().add_to(m)

m.save("outputs/milestone_1_map.html")
m

In [ ]:
"""
**************************************************************************
*
* Author: Bryce Koch
* Email: bryce.koch@my.msutexas.edu
* Label: Project_01_Code
* Title: Missile Geometry 101 - Milestone 2
* Course: CMPS 5993
* Semester: Spring 2026
*
* Description:
* This milestone loads simulated threat data from a JSON file and
* computes the haversine distance from each threat's origin to the
* WDO base. The closest threat is identified and reported to the
* console. All threat origins are plotted as circle markers on the
* Folium map, with the closest threat highlighted in red and all
* others in blue. Each marker includes a popup showing the threat
* ID, type, and distance from base.
*
* Usage:
* Run as a Jupyter notebook cell after completing Milestone 1.
* Requires: m, BASE_LAT, BASE_LON, haversine_km
*
* Files:
* notebook.ipynb     : this notebook cell
* src/threats.json   : simulated threat data
* src/geo_math.py    : haversine_km helper
*
**************************************************************************
"""

import json
from pathlib import Path
import folium

# Path to the generated threat data file
THREATS_PATH = Path("src/threats.json")

with THREATS_PATH.open() as f:
    threats = json.load(f)

print(f"Loaded {len(threats)} threats")

# Compute haversine distance from each threat origin to the WDO base
# and store it back onto the threat dict for use in later milestones
for t in threats:
    dist = haversine_km(
        BASE_LAT,
        BASE_LON,
        t["origin_lat"],
        t["origin_lon"]
    )
    t["distance_km"] = dist

# Identify the single closest threat by shortest distance to base
closest = min(threats, key=lambda x: x["distance_km"])

print("\nClosest Threat:")
print(f"ID: {closest['id']}")
print(f"Type: {closest['type']}")
print(f"Distance: {closest['distance_km']:.2f} km")

# Plot each threat origin as a circle marker on the map
# Red indicates the closest threat, blue indicates all others
for t in threats:
    popup_text = (
        f"ID: {t['id']}<br>"
        f"Type: {t['type']}<br>"
        f"Distance: {t['distance_km']:.1f} km"
    )

    folium.CircleMarker(
        location=(t["origin_lat"], t["origin_lon"]),
        radius=5,
        popup=popup_text,
        color="red" if t["id"] == closest["id"] else "blue",
        fill=True,
        fill_opacity=0.7,
    ).add_to(m)

m.save("outputs/milestone_2_map.html")
m

In [ ]:
"""
**************************************************************************
*
* Author: Bryce Koch
* Email: bryce.koch@my.msutexas.edu
* Label: Project_01_Code
* Title: Missile Geometry 101 - Milestone 3
* Course: CMPS 5993
* Semester: Spring 2026
*
* Description:
* This milestone converts each threat's motion into a visible trajectory
* on the map. For each threat, a destination point is computed using
* bearing and speed over a fixed time interval. Intermediate waypoints
* are generated between the origin and destination and rendered as a
* PolyLine on the Folium map. A green endpoint marker is placed at each
* projected position to clearly show where each threat is headed.
*
* Usage:
* Run as a Jupyter notebook cell after completing Milestones 1-2.
* Requires: threats, m, destination_point, trajectory_points
*
* Files:
* notebook.ipynb     : this notebook cell
* src/geo_math.py    : destination_point, trajectory_points helpers
*
**************************************************************************
"""

from src.geo_math import destination_point, trajectory_points
import folium

# Duration in minutes used to project each threat's position forward
FIXED_TIME_MIN = 20

for t in threats:
    # Compute how far the threat travels in FIXED_TIME_MIN at its speed
    distance_km = t["speed_kmh"] * (FIXED_TIME_MIN / 60.0)

    # Compute the projected destination from origin using bearing and distance
    dest_lat, dest_lon = destination_point(
        t["origin_lat"],
        t["origin_lon"],
        t["bearing_deg"],
        distance_km
    )

    # Store destination back onto the threat dict for use in later milestones
    t["dest_lat"] = dest_lat
    t["dest_lon"] = dest_lon

    # Generate intermediate waypoints every 2 minutes along the trajectory
    # Smaller step_min produces a smoother line on the map
    pts = trajectory_points(
        t["origin_lat"],
        t["origin_lon"],
        t["bearing_deg"],
        t["speed_kmh"],
        FIXED_TIME_MIN,
        step_min=2.0
    )

    # Draw the trajectory as a line connecting all waypoints
    folium.PolyLine(
        pts,
        weight=2,
        opacity=0.7,
        popup=f"{t['id']} trajectory"
    ).add_to(m)

    # Mark the projected endpoint so the threat's heading is visually clear
    folium.CircleMarker(
        location=(dest_lat, dest_lon),
        radius=4,
        color="green",
        fill=True,
        fill_opacity=0.8,
        popup=f"{t['id']} projected position"
    ).add_to(m)

m.save("outputs/milestone_3_map.html")
m

In [ ]:
"""
**************************************************************************
*
* Author: Bryce Koch
* Email: bryce.koch@my.msutexas.edu
* Label: Project_01_Code
* Title: Missile Geometry 101 - Milestone 4
* Course: CMPS 5993
* Semester: Spring 2026
*
* Description:
* This milestone determines which countries are intersected by each
* threat's trajectory and whether any trajectory passes within a
* threshold distance of the WDO base. Intersection is computed using
* both a ray-casting point-in-polygon algorithm and a segment-level
* intersection check to catch threats that pass through thin countries
* between trajectory waypoints. Intersected countries are highlighted
* orange on the Folium map, and trajectories are color-coded red if
* they pass near the base.
*
* Usage:
* Run as a Jupyter notebook cell after completing Milestones 1-3.
* Requires: threats, closest, BASE_LAT, BASE_LON, trajectory_points
*
* Files:
* notebook.ipynb         : this notebook cell
* src/geo_math.py        : haversine_km, trajectory_points helpers
* data/countries.geojson : world country polygons
*
**************************************************************************
"""

import json
from pathlib import Path
import folium
from src.geo_math import haversine_km, trajectory_points

# GeoJSON property field used to identify country names
COUNTRY_FIELD = "ADMIN"
COUNTRIES_PATH = Path("data/countries.geojson")

# Distance threshold in km — trajectories closer than this are flagged near_base
BASE_THRESHOLD_KM = 300

# Time step in minutes between trajectory waypoints
STEP_MIN = 5.0

with COUNTRIES_PATH.open() as f:
    countries_geojson = json.load(f)

print("Loaded countries:", len(countries_geojson["features"]))

# Verify the expected country name field exists before processing
sample_props = countries_geojson["features"][0]["properties"]
assert COUNTRY_FIELD in sample_props, (
    f"Field '{COUNTRY_FIELD}' not found in GeoJSON. "
    f"Available fields: {list(sample_props.keys())}"
)

# Accumulates names of all countries intersected by any trajectory
intersected_countries = set()


def point_in_polygon(x, y, polygon):
    """
    Point In Polygon

    Description:
    Determines whether a given point lies inside a polygon using the
    ray-casting algorithm. A horizontal ray is cast from the point and
    the number of polygon edge crossings is counted — odd means inside.

    Params:
    x (float)        : Longitude of the point to test.
    y (float)        : Latitude of the point to test.
    polygon (list)   : List of (lon, lat) coordinate pairs forming the
                       polygon exterior ring.

    Returns:
    bool: True if the point is inside the polygon, False otherwise.
    """
    n = len(polygon)
    inside = False
    px, py = x, y
    for i in range(n):
        j = (i + 1) % n
        xi, yi = polygon[i]
        xj, yj = polygon[j]

        # Check if the ray crosses this polygon edge
        # Small epsilon prevents division by zero on horizontal edges
        if ((yi > py) != (yj > py)) and (px < (xj - xi) * (py - yi) / (yj - yi + 1e-12) + xi):
            inside = not inside
    return inside


def segment_intersects_polygon(p1, p2, polygon):
    """
    Segment Intersects Polygon

    Description:
    Checks whether a line segment defined by two points intersects any
    edge of a polygon. Uses the counter-clockwise (CCW) cross product
    method to detect edge crossings without computing exact intersections.
    This catches cases where a trajectory passes through a thin country
    entirely between two waypoints.

    Params:
    p1 (tuple)      : (lon, lat) start point of the segment.
    p2 (tuple)      : (lon, lat) end point of the segment.
    polygon (list)  : List of (lon, lat) coordinate pairs forming the
                      polygon exterior ring.

    Returns:
    bool: True if the segment intersects any polygon edge, False otherwise.
    """
    def ccw(A, B, C):
        # Returns True if points A, B, C are in counter-clockwise order
        return (C[1]-A[1]) * (B[0]-A[0]) > (B[1]-A[1]) * (C[0]-A[0])

    def segments_intersect(A, B, C, D):
        # Two segments AB and CD intersect if their endpoints straddle each other
        return ccw(A,C,D) != ccw(B,C,D) and ccw(A,B,C) != ccw(A,B,D)

    n = len(polygon)
    for i in range(n):
        j = (i + 1) % n
        if segments_intersect(p1, p2, polygon[i], polygon[j]):
            return True
    return False


m = folium.Map(location=[BASE_LAT, BASE_LON], zoom_start=4)

# Mark the WDO base location as a reference point on the map
folium.Marker(
    location=(BASE_LAT, BASE_LON),
    popup="WDO Base",
    icon=folium.Icon(color="black", icon="star")
).add_to(m)

# Plot each threat's origin — red for closest, blue for all others
for t in threats:
    folium.CircleMarker(
        location=(t["origin_lat"], t["origin_lon"]),
        radius=5,
        color="red" if t["id"] == closest["id"] else "blue",
        fill=True,
        fill_opacity=0.7,
        popup=f"{t['id']} origin"
    ).add_to(m)

for t in threats:
    # Generate waypoints along the trajectory at fixed time intervals
    pts_latlon = trajectory_points(
        t["origin_lat"],
        t["origin_lon"],
        t["bearing_deg"],
        t["speed_kmh"],
        t["duration_min"],
        step_min=STEP_MIN
    )

    # Folium uses (lat, lon) but intersection math uses (lon, lat)
    traj_coords = [(lon, lat) for lat, lon in pts_latlon]

    # Find the closest any trajectory point gets to the WDO base
    min_dist = min(
        haversine_km(BASE_LAT, BASE_LON, lat, lon)
        for lat, lon in pts_latlon
    )
    t["near_base"] = min_dist <= BASE_THRESHOLD_KM

    for feature in countries_geojson["features"]:
        geom = feature["geometry"]

        # Normalize both Polygon and MultiPolygon to a list of polygons
        if geom["type"] == "Polygon":
            polygons = [geom["coordinates"]]
        elif geom["type"] == "MultiPolygon":
            polygons = geom["coordinates"]
        else:
            continue

        for poly in polygons:
            exterior = poly[0]  # Outer ring only, ignoring interior holes
            hit = False

            # Test each trajectory segment against the polygon
            # Combining segment and point checks catches all intersection cases
            for i in range(len(traj_coords) - 1):
                p1, p2 = traj_coords[i], traj_coords[i + 1]
                if segment_intersects_polygon(p1, p2, exterior) or point_in_polygon(p1[0], p1[1], exterior):
                    hit = True
                    break

            if hit:
                country_id = feature["properties"][COUNTRY_FIELD]
                intersected_countries.add(country_id)
                break

    # Color trajectory red if it passes near the base, blue otherwise
    folium.PolyLine(
        pts_latlon,
        weight=2,
        opacity=0.7,
        color="red" if t["near_base"] else "blue",
        popup=f"{t['id']} trajectory"
    ).add_to(m)


def style_function(feature):
    """
    Style Function

    Description:
    Folium style callback that highlights countries intersected by any
    threat trajectory in orange, and renders all others in gray.

    Params:
    feature (dict): A GeoJSON feature dict passed by Folium internally.

    Returns:
    dict: A style properties dictionary consumed by Folium's GeoJson layer.
    """
    country_name = feature["properties"][COUNTRY_FIELD]
    if country_name in intersected_countries:
        return {
            "fillColor": "orange",
            "color": "orange",
            "weight": 1,
            "fillOpacity": 0.5
        }
    else:
        return {
            "fillColor": "gray",
            "color": "gray",
            "weight": 1,
            "fillOpacity": 0.2
        }


folium.GeoJson(
    countries_geojson,
    style_function=style_function,
    name="Intersected Countries"
).add_to(m)

# Print IDs of any threats passing within the base threshold distance
print("\nThreats passing within threshold:")
for t in threats:
    if t["near_base"]:
        print(t["id"])

m.save("outputs/milestone_4_map.html")
m

In [ ]:
"""
**************************************************************************
*
* Author: Bryce Koch
* Email: bryce.koch@my.msutexas.edu
* Label: Project_01_Code
* Title: Missile Geometry 101 - Milestone 5
* Course: CMPS 5993
* Semester: Spring 2026
*
* Description:
* This milestone generates damage zone buffers around each threat's
* projected endpoint and determines which countries fall within those
* zones. Buffer size varies by threat type. Each affected country is
* assigned a severity rating (CRITICAL, HIGH, MEDIUM, LOW) based on
* proximity to the WDO base and population significance. Results are
* displayed as a markdown table and rendered on the Folium map from
* previous milestones.
*
* Usage:
* Run as a Jupyter notebook cell after completing Milestones 1-4.
* Requires: threats, countries_geojson, m, BASE_LAT, BASE_LON,
*           COUNTRY_FIELD, STEP_MIN, trajectory_points
*
* Files:
* notebook.ipynb     : this notebook cell
* src/geo_math.py    : trajectory_points helper
* data/countries.geojson : world country polygons
*
**************************************************************************
"""

from shapely.geometry import LineString, Point
from shapely.ops import transform
import pyproj
import os

# Buffer radius in kilometers for each threat type
# Larger values reflect greater destructive potential
BUFFER_SIZES_KM = {
    "alien": 500,
    "orbital": 300,
    "airborne": 150,
    "kaiju": 100
}

# Countries considered highly populated for severity classification
# A hit on any of these escalates severity to HIGH
POPULATED_COUNTRIES = {
    "United States", "China", "India", "Brazil", "Russia",
    "Japan", "Germany", "United Kingdom", "France", "Canada",
    "Australia", "Mexico", "South Korea", "Indonesia", "Nigeria"
}


def compute_severity(country_name, buffer_geom, base_lat, base_lon):
    """
    Compute Severity

    Description:
    Determines the severity level of a threat's damage zone based on
    whether it overlaps the WDO base, strikes a populated country,
    or hits any land at all.

    Params:
    country_name (str)   : Name of the affected country.
    buffer_geom (Polygon): Shapely polygon representing the damage zone.
    base_lat (float)     : Latitude of the WDO base.
    base_lon (float)     : Longitude of the WDO base.

    Returns:
    str: One of "CRITICAL", "HIGH", or "MEDIUM".
    """
    base_point = Point(base_lon, base_lat)

    # CRITICAL if the damage zone directly overlaps the WDO base
    if buffer_geom.contains(base_point):
        return "CRITICAL"

    # HIGH if the country is on the populated countries list
    if country_name in POPULATED_COUNTRIES:
        return "HIGH"

    # MEDIUM for any other land intersection
    return "MEDIUM"


# Accumulates one record per country/threat intersection
damage_records = []

for t in threats:
    # Generate lat/lon waypoints along this threat's trajectory
    pts_latlon = trajectory_points(
        t["origin_lat"],
        t["origin_lon"],
        t["bearing_deg"],
        t["speed_kmh"],
        t["duration_min"],
        step_min=STEP_MIN
    )

    # Last point in the trajectory is the projected impact location
    endpoint = pts_latlon[-1]
    buffer_km = BUFFER_SIZES_KM[t["type"]]
    buffer_m = buffer_km * 1000

    # Use azimuthal equidistant projection centered on the endpoint
    # so the buffer radius is accurate in meters, not distorted degrees
    wgs84 = pyproj.CRS("EPSG:4326")
    aeqd = pyproj.CRS(proj="aeqd", lat_0=endpoint[0], lon_0=endpoint[1], datum="WGS84")
    to_aeqd = pyproj.Transformer.from_crs(wgs84, aeqd, always_xy=True).transform
    to_wgs84 = pyproj.Transformer.from_crs(aeqd, wgs84, always_xy=True).transform

    # Project endpoint to AEQD, buffer it, then reproject back to WGS84
    center = Point(endpoint[1], endpoint[0])
    center_proj = transform(to_aeqd, center)
    buffer_proj = center_proj.buffer(buffer_m)
    buffer_wgs84 = transform(to_wgs84, buffer_proj)

    # Track whether this threat's buffer intersects any land
    threat_hit_land = False

    for feature in countries_geojson["features"]:
        geom = feature["geometry"]

        # Normalize both Polygon and MultiPolygon to a list of polygons
        if geom["type"] == "Polygon":
            polygons = [geom["coordinates"]]
        elif geom["type"] == "MultiPolygon":
            polygons = geom["coordinates"]
        else:
            continue

        for poly in polygons:
            exterior = poly[0]  # Outer ring only, ignoring holes
            country_shape = LineString(exterior)

            # Check if the buffer overlaps the country's boundary or interior
            if buffer_wgs84.intersects(country_shape) or buffer_wgs84.contains(
                country_shape.centroid if hasattr(country_shape, 'centroid') else Point(exterior[0])
            ):
                threat_hit_land = True
                damage_records.append({
                    "country": feature["properties"][COUNTRY_FIELD],
                    "threat_type": t["type"],
                    "severity": compute_severity(
                        feature["properties"][COUNTRY_FIELD],
                        buffer_wgs84,
                        BASE_LAT,
                        BASE_LON
                    )
                })
                break

    # If no land was hit, record an ocean impact with LOW severity
    if not threat_hit_land:
        damage_records.append({
            "country": "N/A (Open Ocean)",
            "threat_type": t["type"],
            "severity": "LOW"
        })

    # Convert buffer exterior coords from (lon, lat) to (lat, lon) for Folium
    buffer_coords = [(lat, lon) for lon, lat in buffer_wgs84.exterior.coords]

    # Color-code buffer polygons by threat type for visual clarity
    folium.Polygon(
        locations=buffer_coords,
        color={"alien": "purple", "orbital": "red", "airborne": "orange", "kaiju": "brown"}[t["type"]],
        fill=True,
        fill_opacity=0.3,
        weight=1,
        popup=f"{t['id']} damage zone ({buffer_km} km)"
    ).add_to(m)

# Remove duplicate country/threat_type combinations before printing
seen = set()
unique_records = []
for r in damage_records:
    key = (r["country"], r["threat_type"])
    if key not in seen:
        seen.add(key)
        unique_records.append(r)

# Print results as a markdown-formatted table for use in README
print(f"| {'country':<35} | {'threat_type':<12} | {'severity':<10} |")
print(f"|{'-'*37}|{'-'*14}|{'-'*12}|")
for r in unique_records:
    print(f"| {r['country']:<35} | {r['threat_type']:<12} | {r['severity']:<10} |")

# Save map to outputs folder, creating it if it doesn't exist
os.makedirs("outputs", exist_ok=True)
m.save("outputs/milestone_5_map.html")
m